# One tile: metadata, contacts, contour check, design ↔ SEM matching

`read_tile_index(DATA_DIR)` (remote agent, task T001) reads the metadata CSV and pairs each
SEM image with its `.oas` in `Contour/`. This notebook looks at one tile:
1. the tile table;
2. the SEM image;
3. the tile's design and SEM contacts in **mask coordinates** (tile-local + tile centre);
4. a **contour check** of one contact: Otsu, gradient-max refined and `.oas` contours;
5. design ↔ SEM **matching**: only contacts found in both are kept, with their offset.

Positions are shown in µm, position errors in nm.

In [ ]:
DATA_DIR = r"A:\Stitch_monitor\In_cell_without_PTM_20260914\260910_1143_Norm"
TILE = 0                # which tile to look at (index in the table)
LAYER = (100, 0)        # (layer, datatype) of the contacts in the .oas
TONE_REVERSED = False   # True if this tile's .oas draws the area AROUND the holes
REFINE_EDGES = True     # refine Otsu contours to the maximum intensity gradient
MAX_MATCH_NM = 10.0     # pair design/SEM contacts closer than this
CONTACT = 0             # which SEM contact to inspect in the contour check

In [ ]:
%matplotlib widget
import dataclasses

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from affine_ransac.features.contact import detect_contacts
from affine_ransac.features.edges import refine_edges, sample_rays
from affine_ransac.geometry.frames import pixel_to_tile_nm, tile_nm_to_pixel
from affine_ransac.io.design import load_layout, read_contacts, read_contacts_tone_reversed, read_polygons
from affine_ransac.io.metadata import read_tile_index
from affine_ransac.io.sem_image import load_sem_image
from affine_ransac.matching import match_points

## 1. Tile table (µm)

In [ ]:
tiles = read_tile_index(DATA_DIR)

table = pd.DataFrame([dataclasses.asdict(t) for t in tiles]).drop(columns="extra")
table["image_path"] = table.image_path.map(lambda p: p.name)
table["oas_path"] = table.oas_path.map(lambda p: p.name if p else None)
for column in ["center_x_nm", "center_y_nm", "fov_x_nm", "fov_y_nm", "stage_x_nm", "stage_y_nm"]:
    table[column.replace("_nm", "_um")] = table.pop(column) / 1000

print(f"{len(tiles)} tiles, {table.oas_path.isna().sum()} without an .oas file")
table

## 2. The SEM image

In [ ]:
tile = tiles[TILE]
image = load_sem_image(tile.image_path)
pixel_size_nm = tile.fov_x_nm / image.shape[1]
print(f"Tile {tile.tile_id}: {image.shape[1]} x {image.shape[0]} px, pixel size {pixel_size_nm:.4f} nm")

plt.figure(figsize=(7, 7))
plt.imshow(image, cmap="gray")
plt.title(f"{tile.image_path.name}  (x right, y down, px)")
plt.show()

## 3. Design and SEM contacts in mask coordinates

Design: `.oas` positions (centred on (0, 0)) + tile centre.
SEM: Otsu contacts, refined to the gradient maximum if `REFINE_EDGES`, → tile-local nm → + tile centre.

In [ ]:
layout = load_layout(tile.oas_path)
if TONE_REVERSED:
    design_local, design_sizes = read_contacts_tone_reversed(layout, *LAYER, frame_nm=(tile.fov_x_nm, tile.fov_y_nm))
else:
    design_local, design_sizes = read_contacts(layout, *LAYER)

otsu = detect_contacts(image)
found = refine_edges(image, otsu) if REFINE_EDGES else otsu
sem_local = pixel_to_tile_nm(found.centers, image.shape, pixel_size_nm)

center = (tile.center_x_nm, tile.center_y_nm)
design_mask = design_local + center
sem_mask = sem_local + center
print(f"Tile centre (mask): x = {center[0] / 1000:.4f} um, y = {center[1] / 1000:.4f} um")
print(f"Design contacts: {len(design_mask)}, SEM contacts: {len(sem_mask)}")

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(design_mask[:, 0] / 1000, design_mask[:, 1] / 1000, marker="s", facecolors="none", edgecolors="tab:blue", label="design (.oas)")
ax.scatter(sem_mask[:, 0] / 1000, sem_mask[:, 1] / 1000, marker="+", color="tab:red", label="SEM")
ax.set_aspect("equal")
ax.set_xlabel("mask x (µm)")
ax.set_ylabel("mask y (µm)")
ax.legend()
plt.show()

## 4. Contour check of one contact

Left: the SEM image around contact `CONTACT`, with the Otsu contour (green), the
gradient-max refined contour (red) and the `.oas` contour (blue).
Right: intensity (and its gradient) along a ray from the contact centre to the right (+x),
with the edge positions of the three contours on that ray.

In [ ]:
k = CONTACT
c = otsu.centers[k]
otsu_radius = np.sqrt(otsu.areas[k] / np.pi)
half = 2.5 * otsu_radius
refined = refine_edges(image, otsu)  # always computed here, for the comparison


def closed(points):
    """x and y arrays of a polygon with its first point repeated at the end, for plotting."""
    return np.vstack([points, points[:1]]).T


design_polygons_px = [tile_nm_to_pixel(p, image.shape, pixel_size_nm) for p in read_polygons(layout, *LAYER)]

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image, cmap="gray")
for i, polygon in enumerate(design_polygons_px):
    ax.plot(*closed(polygon), color="tab:blue", lw=1, label=".oas" if i == 0 else None)
ax.plot(*closed(otsu.contours[k]), color="tab:green", lw=1, label="Otsu")
ax.plot(*closed(refined.contours[k]), color="tab:red", lw=1, label="gradient max")
ax.set_xlim(c[0] - half, c[0] + half)
ax.set_ylim(c[1] + half, c[1] - half)  # y points down
ax.set_title(f"Contact {k}")
ax.legend()
plt.show()

In [ ]:
radii = np.arange(0, half, 0.25)
profile = sample_rays(image, c, np.array([0.0]), radii)[0]

# Edge positions on this ray (angle 0 = +x), in px from the Otsu centre.
otsu_edge = otsu.contours[k][:, 0].max() + 0.5 - c[0]   # outer side of the last Otsu pixel
refined_edge = refined.contours[k][0, 0] - c[0]          # the first refined ray is at angle 0
design_px = tile_nm_to_pixel(design_local, image.shape, pixel_size_nm)
j = np.argmin(np.linalg.norm(design_px - c, axis=1))     # the design contact nearest to this one
design_edge = design_px[j, 0] + design_sizes[j, 0] / 2 / pixel_size_nm - c[0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(radii * pixel_size_nm, profile, color="black", label="intensity")
gradient_ax = ax.twinx()
gradient_ax.plot(radii * pixel_size_nm, np.gradient(profile, radii), color="gray", lw=0.8)
gradient_ax.set_ylabel("gradient (grey level / px, gray line)")
for edge, color, name in [(otsu_edge, "tab:green", "Otsu"), (refined_edge, "tab:red", "gradient max"), (design_edge, "tab:blue", ".oas")]:
    ax.axvline(edge * pixel_size_nm, color=color, label=f"{name}: {edge * pixel_size_nm:.1f} nm")
ax.set_xlabel("distance from contact centre along +x (nm)")
ax.set_ylabel("intensity")
ax.legend(loc="lower right")
plt.show()

## 5. Match design ↔ SEM

Each design contact is paired with its SEM contact (mutual nearest, closer than
`MAX_MATCH_NM`). Contacts found in only one of the two are left out and circled.
`dx_nm, dy_nm` = SEM − design, before any affine correction.

In [ ]:
i_design, i_sem = match_points(design_mask, sem_mask, MAX_MATCH_NM)
pairs = pd.DataFrame({
    "x_um": design_mask[i_design, 0] / 1000, "y_um": design_mask[i_design, 1] / 1000,
    "dx_nm": sem_mask[i_sem, 0] - design_mask[i_design, 0],
    "dy_nm": sem_mask[i_sem, 1] - design_mask[i_design, 1],
})
unmatched_design = design_mask[np.setdiff1d(np.arange(len(design_mask)), i_design)]
unmatched_sem = sem_mask[np.setdiff1d(np.arange(len(sem_mask)), i_sem)]

print(f"Matched pairs: {len(pairs)}")
print(f"Design only: {len(unmatched_design)}, SEM only: {len(unmatched_sem)}")
print(f"Offset: dx mean {pairs.dx_nm.mean():.2f} nm (std {pairs.dx_nm.std():.2f}), "
      f"dy mean {pairs.dy_nm.mean():.2f} nm (std {pairs.dy_nm.std():.2f})")

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(pairs.x_um, pairs.y_um, marker=".", color="gray", label="matched")
ax.scatter(unmatched_design[:, 0] / 1000, unmatched_design[:, 1] / 1000, s=120, facecolors="none", edgecolors="tab:blue", label="design only")
ax.scatter(unmatched_sem[:, 0] / 1000, unmatched_sem[:, 1] / 1000, s=120, facecolors="none", edgecolors="tab:red", label="SEM only")
ax.set_aspect("equal")
ax.set_xlabel("mask x (µm)")
ax.set_ylabel("mask y (µm)")
ax.legend()
plt.show()
pairs